In [1]:
import os
import gc
import time
import sqlite3
import subprocess
from datetime import datetime
import pyarrow.parquet as pq
from nested_pandas.datasets import generate_data

compressions = [None, "snappy", "gzip", "brotli", "lz4", "zstd"]


def GENERATE_DATA(b, n, file_path, algo, is_flat):
    """Generates synthetic nested or flat data and writes it to a Parquet file."""
    nf = generate_data(b, n, seed=1)[["nested"]]
    
    if is_flat:
        # .to_lists() already returns an object that can be saved directly to Parquet
        df = nf["nested"].to_lists()
        df.to_parquet(file_path, compression=algo)
    else:
        # Save as a native nested Parquet structure
        nf.to_parquet(file_path, compression=algo)


def SAVE_ALGO_TO_SQLITE3(sqlite3_path, file_obj, algo_key):
    """Saves individual benchmark runs for a SINGLE algorithm to the database."""
    conn = sqlite3.connect(sqlite3_path)
    cursor = conn.cursor()

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS benchmark_results (
            file_type TEXT,
            base_rows INTEGER,
            nested_rows INTEGER,
            compression TEXT,
            file_size_bytes INTEGER,
            run_number INTEGER,
            run_timestamp DATETIME,
            run_time_sec REAL
        )
    """)

    file_type = "FlatFile" if file_obj.is_flat else "NestedFile"
    file_size = file_obj.file_sizes.get(algo_key, 0)

    if algo_key in file_obj.benchmarking_times:
        for run_data in file_obj.benchmarking_times[algo_key]:
            run_number, timestamp, run_time = run_data

            cursor.execute(
                """
                INSERT INTO benchmark_results 
                (file_type, base_rows, nested_rows, compression, file_size_bytes, run_number, run_timestamp, run_time_sec)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?)
            """,
                (
                    file_type,
                    file_obj.b,
                    file_obj.n,
                    algo_key,
                    file_size,
                    run_number,
                    timestamp,
                    run_time,
                ),
            )

    conn.commit()
    conn.close()


class File:
    def __init__(self, b: int, n: int, path: str) -> None:
        self.b = b
        self.n = n
        self.order = b * n
        self.file_path = path
        self.file_exists = False

        self.file_sizes = {}
        self.benchmarking_times = {
            "None": [],
            "snappy": [],
            "gzip": [],
            "brotli": [],
            "lz4": [],
            "zstd": [],
        }

        self.current_algo = ""
        self.is_flat = False

    def generate_data(self, algo):
        GENERATE_DATA(self.b, self.n, self.file_path, algo, self.is_flat)
        self.file_exists = os.path.exists(self.file_path)

        algo_key = "None" if algo is None else algo
        self.file_sizes[algo_key] = (
            os.path.getsize(self.file_path) if self.file_exists else 0
        )

    def pre_benchmark(self):
        """Loads the file into the OS page cache using vmtouch."""
        if self.file_exists:
            subprocess.run(
                ["vmtouch", "-t", self.file_path], check=True, stdout=subprocess.DEVNULL
            )

    def run_benchmark_loop(self, db_path, num_runs=10):
        """Executes the complete granular lifecycle per algorithm sequentially."""
        for algo in compressions:
            self.current_algo = "None" if algo is None else algo
            print(f"      -> Processing Compression: {self.current_algo.upper()}")

            # 1 & 2. Choose algorithm & generate file
            self.generate_data(algo)

            # 3. Load file completely into OS page cache
            self.pre_benchmark()

            # 4. Benchmark the file for the specified number of runs
            for run_idx in range(1, num_runs + 1):
                run_timestamp = datetime.now().isoformat()

                start_time = time.perf_counter()
                table = pq.read_table(self.file_path, use_threads=True)
                run_time = time.perf_counter() - start_time

                del table
                gc.collect()

                self.benchmarking_times[self.current_algo].append(
                    (run_idx, run_timestamp, run_time)
                )

            # 5 & 6. Clear cache & delete file immediately after benchmarking
            self.post_benchmark()

            # 7. Save this specific algorithm's run data immediately to SQL
            SAVE_ALGO_TO_SQLITE3(db_path, self, self.current_algo)

    def post_benchmark(self):
        """Evicts the file from the OS page cache and deletes it from disk."""
        if self.file_exists:
            subprocess.run(
                ["vmtouch", "-e", self.file_path], check=True, stdout=subprocess.DEVNULL
            )
            os.remove(self.file_path)
            self.file_exists = False


class FlatFile(File):
    def __init__(self, b: int, n: int, path: str) -> None:
        super().__init__(b, n, path)
        self.is_flat = True


class NestedFile(File):
    def __init__(self, b: int, n: int, path: str) -> None:
        super().__init__(b, n, path)
        self.is_flat = False


# ==========================================
# Main Execution Pipeline
# ==========================================
if __name__ == "__main__":
    db_path = "benchmarks.db"

    size_configs = [
        # ==========================================
        # 10^2 (100 items) - Micro Scale
        # ==========================================
        (10, 10),
        # ==========================================
        # 10^3 (1,000 items)
        # ==========================================
        (10, 100),  # Extremely Deep
        (100, 10),  # Extremely Wide
        # ==========================================
        # 10^4 (10,000 items)
        # ==========================================
        (10, 1000),  # Extremely Deep
        (100, 100),  # Balanced
        (1000, 10),  # Extremely Wide
        # ==========================================
        # 10^5 (100,000 items)
        # ==========================================
        (10, 10000),  # Extremely Deep
        (100, 1000),  # Deep
        (1000, 100),  # Wide
        (10000, 10),  # Extremely Wide
        # ==========================================
        # 10^6 (1 Million items) ~ Approx 8 MB uncompressed
        # ==========================================
        (10, 100000),  # Extremely Deep
        (100, 10000),  # Deep
        (1000, 1000),  # Balanced
        (10000, 100),  # Wide
        (100000, 10),  # Extremely Wide
        # ==========================================
        # 10^7 (10 Million items) ~ Approx 80 MB uncompressed
        # ==========================================
        (10, 1000000),  # Extremely Deep
        (100, 100000),  # Deep
        (1000, 10000),  # Slightly Deep
        (10000, 1000),  # Slightly Wide
        (100000, 100),  # Wide
        (1000000, 10),  # Extremely Wide
        # ==========================================
        # 10^8 (100 Million items) ~ Approx 800 MB uncompressed
        # ==========================================
        (10, 10000000),  # Extremely Deep
        (100, 1000000),  # Deep
        (1000, 100000),  # Slightly Deep
        (10000, 10000),  # Balanced
        (100000, 1000),  # Slightly Wide
        (1000000, 100),  # Wide
        (10000000, 10),  # Extremely Wide
        # ==========================================
        # 10^9 (1 Billion items) ~ Approx 8 GB uncompressed
        # ==========================================
        # Uncomment below if you have enough system RAM!
        # (10, 100000000),  # Extremely Deep
        # (100, 10000000),  # Deep
        # (1000, 1000000),  # Slightly Deep
        # (31622, 31622),   # Balanced (sqrt of 10^9)
        # (1000000, 1000),  # Slightly Wide
        # (10000000, 100),  # Wide
        # (100000000, 10),  # Extremely Wide
    ]

    total_configs = len(size_configs)

    for idx, (base_rows, nested_rows) in enumerate(size_configs, 1):
        # Start timer for this entire configuration matrix
        config_start_time = time.perf_counter()
        
        # Calculate dynamic run limits
        total_elements = base_rows * nested_rows
        if total_elements < 10**4:
            current_num_runs = 50
        elif total_elements <= 10**6:
            current_num_runs = 10
        else:
            current_num_runs = 1

        print(f"\n========================================================")
        print(f"[{idx}/{total_configs}] Processing Dataset Configuration: {base_rows}x{nested_rows}")
        print(f"Executing {current_num_runs} runs per compression algo (Total Elements: {total_elements:,})")
        print(f"========================================================")

        # --- Task 1: Pipeline loop for Flat representation ---
        print("  -> Starting FlatFile Sequence...")
        flat = FlatFile(base_rows, nested_rows, "temp_flat.parquet")
        flat.run_benchmark_loop(db_path, num_runs=current_num_runs)

        # --- Task 2: Pipeline loop for Native Nested representation ---
        print("  -> Starting NestedFile Sequence...")
        nested = NestedFile(base_rows, nested_rows, "temp_nested.parquet")
        nested.run_benchmark_loop(db_path, num_runs=current_num_runs)

        # Stop timer and report duration
        config_end_time = time.perf_counter()
        elapsed_sec = config_end_time - config_start_time
        
        print(f"--------------------------------------------------------")
        print(f" [✓] Configuration {base_rows}x{nested_rows} completed in {elapsed_sec:.2f} seconds.")
        print(f"--------------------------------------------------------")

    print("\nAll tasks in size matrix successfully processed. All run metrics saved directly to SQLite3.")


[1/28] Processing Dataset Configuration: 10x10
Executing 50 runs per compression algo (Total Elements: 100)
  -> Starting FlatFile Sequence...
      -> Processing Compression: NONE
      -> Processing Compression: SNAPPY
      -> Processing Compression: GZIP
      -> Processing Compression: BROTLI
      -> Processing Compression: LZ4
      -> Processing Compression: ZSTD
  -> Starting NestedFile Sequence...
      -> Processing Compression: NONE
      -> Processing Compression: SNAPPY
      -> Processing Compression: GZIP
      -> Processing Compression: BROTLI
      -> Processing Compression: LZ4
      -> Processing Compression: ZSTD
--------------------------------------------------------
 [✓] Configuration 10x10 completed in 29.03 seconds.
--------------------------------------------------------

[2/28] Processing Dataset Configuration: 10x100
Executing 50 runs per compression algo (Total Elements: 1,000)
  -> Starting FlatFile Sequence...
      -> Processing Compression: NONE
     

In [2]:
import os
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

DB_PATH = "benchmarks.db"
OUTPUT_DIR = "benchmark_graphs"

def load_data():
    """Loads and preprocesses data from SQLite3."""
    if not os.path.exists(DB_PATH):
        raise FileNotFoundError(f"Database not found at {DB_PATH}. Run the benchmark script first.")
        
    conn = sqlite3.connect(DB_PATH)
    df = pd.read_sql_query("SELECT * FROM benchmark_results", conn)
    conn.close()
    
    # Create helper columns
    df['total_elements'] = df['base_rows'] * df['nested_rows']
    df['shape_label'] = df['base_rows'].astype(str) + "x" + df['nested_rows'].astype(str)
    df['file_size_mb'] = df['file_size_bytes'] / (1024 * 1024)
    
    return df

def plot_scaling_performance(df):
    """Graph 1: Flat vs Nested Read Time as Data Scales (Uncompressed Only)"""
    print("Generating: Scaling Performance Plot...")
    
    # Filter for uncompressed runs to get raw PyArrow structure overhead
    uncompressed = df[df['compression'] == 'None'].copy()
    
    # Group by total elements and file type, taking the mean time
    grouped = uncompressed.groupby(['total_elements', 'file_type'])['run_time_sec'].mean().unstack()
    
    plt.figure(figsize=(10, 6))
    if 'FlatFile' in grouped.columns:
        plt.plot(grouped.index, grouped['FlatFile'], marker='o', linewidth=2, label='FlatFile')
    if 'NestedFile' in grouped.columns:
        plt.plot(grouped.index, grouped['NestedFile'], marker='s', linewidth=2, label='NestedFile')
    
    plt.xscale('log')
    plt.yscale('log')
    plt.title('PyArrow Read Time: Flat vs Nested (Uncompressed)', fontsize=14, pad=15)
    plt.xlabel('Total Elements (Base × Nested) [Log Scale]', fontsize=12)
    plt.ylabel('Average Read Time (Seconds) [Log Scale]', fontsize=12)
    plt.grid(True, which="both", ls="--", alpha=0.5)
    plt.legend(title='File Structure')
    
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "1_scaling_performance.png"), dpi=300)
    plt.close()

def plot_deep_vs_wide(df):
    """Graph 2: Deep vs Wide Performance at a fixed scale (100,000 elements)"""
    print("Generating: Deep vs Wide Plot...")
    
    TARGET_ELEMENTS = 100_000
    subset = df[(df['total_elements'] == TARGET_ELEMENTS) & (df['compression'] == 'None')].copy()
    
    if subset.empty:
        print(f"  -> Skipping Graph 2: No uncompressed data found for {TARGET_ELEMENTS} elements.")
        return
        
    grouped = subset.groupby(['shape_label', 'file_type'])['run_time_sec'].mean().unstack()
    
    # Sort by base rows so it goes from Deep (10x10000) to Wide (10000x10)
    grouped = grouped.loc[sorted(grouped.index, key=lambda x: int(x.split('x')[0]))]
    
    ax = grouped.plot(kind='bar', figsize=(10, 6), width=0.7, color=['#1f77b4', '#ff7f0e'])
    
    plt.title(f'Deep vs Wide Data Structures ({TARGET_ELEMENTS:,} Total Elements)', fontsize=14, pad=15)
    plt.xlabel('Dataset Shape (Base Rows × Nested Rows)', fontsize=12)
    plt.ylabel('Average Read Time (Seconds)', fontsize=12)
    plt.xticks(rotation=0)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.legend(title='File Structure')
    
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "2_deep_vs_wide.png"), dpi=300)
    plt.close()

def plot_compression_comparison(df):
    """Graph 3: Compression Speed vs File Size Tradeoffs"""
    print("Generating: Compression Tradeoff Plot...")
    
    # Find a nicely sized dataset that isn't too small or too massive (e.g., 1,000,000 elements)
    available_sizes = df['total_elements'].unique()
    target_size = 1_000_000 if 1_000_000 in available_sizes else available_sizes[-1]
    
    subset = df[(df['total_elements'] == target_size) & (df['file_type'] == 'NestedFile')].copy()
    
    if subset.empty:
         print(f"  -> Skipping Graph 3: No nested data found for {target_size} elements.")
         return

    grouped = subset.groupby('compression').agg({
        'run_time_sec': 'mean',
        'file_size_mb': 'mean'
    }).sort_values('run_time_sec')
    
    fig, ax1 = plt.subplots(figsize=(10, 6))

    # Bar chart for Read Time
    color = 'tab:blue'
    ax1.set_xlabel('Compression Algorithm', fontsize=12)
    ax1.set_ylabel('Average Read Time (Seconds)', color=color, fontsize=12)
    bars = ax1.bar(grouped.index, grouped['run_time_sec'], color=color, alpha=0.7, width=0.4, align='center')
    ax1.tick_params(axis='y', labelcolor=color)

    # Line chart for File Size on secondary Y axis
    ax2 = ax1.twinx()  
    color = 'tab:red'
    ax2.set_ylabel('File Size (MB)', color=color, fontsize=12)  
    line = ax2.plot(grouped.index, grouped['file_size_mb'], color=color, marker='o', linewidth=2, linestyle='dashed')
    ax2.tick_params(axis='y', labelcolor=color)

    plt.title(f'Compression Read Times vs File Size\n(NestedFile - {target_size:,} Elements)', fontsize=14, pad=15)
    fig.tight_layout()
    plt.grid(axis='y', linestyle=':', alpha=0.5)
    
    plt.savefig(os.path.join(OUTPUT_DIR, "3_compression_tradeoffs.png"), dpi=300)
    plt.close()

if __name__ == "__main__":
    if not os.path.exists(OUTPUT_DIR):
        os.makedirs(OUTPUT_DIR)
        
    print("Loading data from SQLite3...")
    df = load_data()
    
    print(f"Found {len(df)} total benchmark runs.")
    
    plot_scaling_performance(df)
    plot_deep_vs_wide(df)
    plot_compression_comparison(df)
    
    print(f"\nAnalysis complete! Graphs saved to the '{OUTPUT_DIR}' directory.")

Loading data from SQLite3...
Found 3396 total benchmark runs.
Generating: Scaling Performance Plot...
Generating: Deep vs Wide Plot...
Generating: Compression Tradeoff Plot...

Analysis complete! Graphs saved to the 'benchmark_graphs' directory.


In [5]:
import os
import sqlite3
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

DB_PATH = "benchmarks.db"
BASE_OUTPUT_DIR = "benchmark_percentile_analysis"

# Enforce a consistent order for the X-axis across all graphs
COMPRESSION_ORDER = ["uncompressed", "snappy", "gzip", "brotli", "lz4", "zstd"]

def analyze_and_plot():
    if not os.path.exists(DB_PATH):
        raise FileNotFoundError(f"Database not found at {DB_PATH}.")

    # 1. Load Data
    conn = sqlite3.connect(DB_PATH)
    df = pd.read_sql_query("SELECT * FROM benchmark_results", conn)
    conn.close()

    if df.empty:
        print("Database is empty. No graphs to generate.")
        return

    # Clean up 'None' compression names for the plot labels
    df['compression'] = df['compression'].fillna('uncompressed').replace('None', 'uncompressed')

    # 2. Compute Total Elements and Order of Magnitude
    df['total_elements'] = df['base_rows'] * df['nested_rows']
    df['order_of_magnitude'] = np.log10(df['total_elements']).astype(int)

    os.makedirs(BASE_OUTPUT_DIR, exist_ok=True)
    sns.set_theme(style="whitegrid")

    # 3. Iterate over Orders of Magnitude (10^2, 10^3, etc.)
    for oom, oom_group in df.groupby('order_of_magnitude'):
        dir_name = f"10^{oom}_elements"
        dir_path = os.path.join(BASE_OUTPUT_DIR, dir_name)
        os.makedirs(dir_path, exist_ok=True)
        print(f"\n📂 Processing Directory: {dir_name}/")

        # 4. Iterate over specific Base x Nested combinations within this order of magnitude
        combinations = oom_group.groupby(['base_rows', 'nested_rows'])
        
        for (base, nested), combo_df in combinations:
            combo_name = f"{base}x{nested}"
            
            plt.figure(figsize=(14, 7))
            
            # Using Seaborn to plot the grouped box plot
            # whis=[10, 90] sets the whiskers exactly at the 10th and 90th percentiles.
            # Data outside this range (0th to 9th, and 91st to 100th percentiles) are plotted as outlier points.
            # showmeans=True calculates and plots the average for that group.
            ax = sns.boxplot(
                data=combo_df,
                x='compression',
                y='run_time_sec',
                hue='file_type',
                order=COMPRESSION_ORDER,
                whis=[10, 90],        
                showmeans=True,       
                meanprops={
                    "marker": "D", 
                    "markerfacecolor": "white", 
                    "markeredgecolor": "black", 
                    "markersize": "7"
                },
                flierprops={
                    "marker": "o", 
                    "markerfacecolor": "gray", 
                    "markersize": 4, 
                    "alpha": 0.6
                },
                boxprops={"alpha": 0.8}
            )

            # Titles and Labels
            plt.title(
                f"Benchmark Run Times: {combo_name} (10$^{{{oom}}}$ Total Elements)\n"
                f"Box: 25th-75th %ile | Whiskers: 10th & 90th %ile | Outliers: 0th (Min) & 100th (Max) %ile | Diamond: Average",
                fontsize=14, pad=15
            )
            plt.xlabel("Compression Algorithm", fontsize=12, fontweight='bold')
            plt.ylabel("Run Time (Seconds)", fontsize=12, fontweight='bold')
            
            # Adjust legend placement
            plt.legend(title='File Structure', bbox_to_anchor=(1.01, 1), loc='upper left')
            
            plt.tight_layout()
            
            # 5. Save the Graph
            output_filepath = os.path.join(dir_path, f"graph_{combo_name}.png")
            plt.savefig(output_filepath, dpi=300, bbox_inches="tight")
            plt.close()
            
            print(f"  └── Saved: graph_{combo_name}.png (N={len(combo_df)} runs)")

if __name__ == "__main__":
    print("Initializing analysis...")
    analyze_and_plot()
    print(f"\n✅ All graphs successfully generated in the '{BASE_OUTPUT_DIR}' directory.")

Initializing analysis...

📂 Processing Directory: 10^2_elements/
  └── Saved: graph_10x10.png (N=600 runs)

📂 Processing Directory: 10^3_elements/
  └── Saved: graph_10x100.png (N=600 runs)
  └── Saved: graph_100x10.png (N=600 runs)

📂 Processing Directory: 10^4_elements/
  └── Saved: graph_10x1000.png (N=120 runs)
  └── Saved: graph_100x100.png (N=120 runs)
  └── Saved: graph_1000x10.png (N=120 runs)

📂 Processing Directory: 10^5_elements/
  └── Saved: graph_10x10000.png (N=120 runs)
  └── Saved: graph_100x1000.png (N=120 runs)
  └── Saved: graph_1000x100.png (N=120 runs)
  └── Saved: graph_10000x10.png (N=120 runs)

📂 Processing Directory: 10^6_elements/
  └── Saved: graph_10x100000.png (N=120 runs)
  └── Saved: graph_100x10000.png (N=120 runs)
  └── Saved: graph_1000x1000.png (N=120 runs)
  └── Saved: graph_10000x100.png (N=120 runs)
  └── Saved: graph_100000x10.png (N=120 runs)

📂 Processing Directory: 10^7_elements/
  └── Saved: graph_10x1000000.png (N=12 runs)
  └── Saved: graph_

In [4]:
%pip install seaborn


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: /home/ink/DEV/pyarrow-lincc-fw-openastronomy-gsoc26/runner/runner_files/venv/bin/python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [6]:
import os
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DB_PATH = "benchmarks.db"
OUTPUT_IMG = "benchmark_summary_table.png"

def generate_table_image():
    if not os.path.exists(DB_PATH):
        print(f"Database not found at {DB_PATH}.")
        return

    # 1. Load Data
    conn = sqlite3.connect(DB_PATH)
    df = pd.read_sql_query("SELECT * FROM benchmark_results", conn)
    conn.close()

    if df.empty:
        print("No data found in the database.")
        return

    # 2. Data Preparation
    df['total_elements'] = df['base_rows'] * df['nested_rows']
    df['order_of_magnitude'] = "10^" + np.log10(df['total_elements']).astype(int).astype(str)
    df['configuration'] = df['base_rows'].astype(str) + "x" + df['nested_rows'].astype(str)
    df['compression'] = df['compression'].fillna('uncompressed').replace('None', 'uncompressed')

    # Group by the specific configuration and get the average run time
    agg = df.groupby(['order_of_magnitude', 'total_elements', 'configuration', 'compression', 'file_type'])['run_time_sec'].mean().unstack().reset_index()

    # 3. Calculate Performance Difference
    # Formula: ((Flat - Nested) / Flat) * 100
    # -> If Nested is SLOWER (Nested > Flat), result is negative (-)
    # -> If Nested is FASTER (Nested < Flat), result is positive (+)
    agg['% Nested Diff'] = ((agg['FlatFile'] - agg['NestedFile']) / agg['FlatFile']) * 100

    # Sort logically by size, then shape, then algorithm
    agg = agg.sort_values(by=['total_elements', 'configuration', 'compression'])

    # 4. Format for Display
    display_df = agg[['order_of_magnitude', 'configuration', 'compression', '% Nested Diff']].copy()
    
    # Store raw numerical values for color-coding before we convert them to strings
    raw_diffs = display_df['% Nested Diff'].values
    display_df['% Nested Diff'] = display_df['% Nested Diff'].apply(lambda x: f"{x:+.2f}%")
    display_df.columns = ['Order of Magnitude', 'Configuration', 'Compression', '% Nested Faster/Slower']

    # 5. Render Table as an Image Using Matplotlib
    print(f"Rendering table image with {len(display_df)} rows. This might take a few seconds...")
    
    # Dynamically scale the image height based on the number of rows so it doesn't squish
    fig_width = 8
    fig_height = len(display_df) * 0.22 + 1.5
    
    fig, ax = plt.subplots(figsize=(fig_width, fig_height))
    ax.axis('off')
    ax.axis('tight')

    table = ax.table(
        cellText=display_df.values,
        colLabels=display_df.columns,
        loc='center',
        cellLoc='center'
    )

    # 6. Styling the Table
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1.2, 1.5)

    # Iterate through cells to apply colors
    for (row, col), cell in table.get_celld().items():
        # Header Row
        if row == 0:
            cell.set_text_props(weight='bold', color='white', fontsize=11)
            cell.set_facecolor('#4c72b0')
        
        # Target the % Difference Column (Index 3)
        elif col == 3:
            val = raw_diffs[row - 1]
            if val > 0:
                cell.set_text_props(color='green', weight='bold')
                cell.set_facecolor('#e6f2e6') # Light green background
            elif val < 0:
                cell.set_text_props(color='red', weight='bold')
                cell.set_facecolor('#f2e6e6') # Light red background

        # Alternating row colors for readability
        if row > 0 and col != 3:
            if row % 2 == 0:
                cell.set_facecolor('#f8f9fa')

    plt.title("Benchmark Summary: Nested vs Flat Architecture Performance", fontsize=14, fontweight='bold', pad=20)
    
    plt.tight_layout()
    plt.savefig(OUTPUT_IMG, dpi=200, bbox_inches="tight")
    plt.close()
    
    print(f"✅ High-resolution table successfully saved as '{OUTPUT_IMG}'")

if __name__ == "__main__":
    generate_table_image()

Rendering table image with 168 rows. This might take a few seconds...
✅ High-resolution table successfully saved as 'benchmark_summary_table.png'


In [7]:
import os
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DB_PATH = "benchmarks.db"
OUTPUT_DIR = "algorithm_percentage_graphs"

def generate_algorithm_graphs():
    if not os.path.exists(DB_PATH):
        print(f"Database not found at {DB_PATH}. Run the benchmark script first.")
        return

    # 1. Load Data
    conn = sqlite3.connect(DB_PATH)
    df = pd.read_sql_query("SELECT * FROM benchmark_results", conn)
    conn.close()

    if df.empty:
        print("No data found in the database.")
        return

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # 2. Data Preparation
    df['compression'] = df['compression'].fillna('uncompressed').replace('None', 'uncompressed')
    df['total_elements'] = df['base_rows'] * df['nested_rows']
    df['order_of_magnitude'] = np.log10(df['total_elements']).astype(int)
    
    # Calculate the mean run time for each exact configuration
    agg = df.groupby(['compression', 'order_of_magnitude', 'total_elements', 'base_rows', 'nested_rows', 'file_type'])['run_time_sec'].mean().unstack().reset_index()

    # Calculate Percentage Difference
    # Formula: ((Flat - Nested) / Flat) * 100
    # (+) Green -> Nested is FASTER (takes less time than Flat)
    # (-) Red   -> Nested is SLOWER (takes more time than Flat)
    agg['percent_diff'] = ((agg['FlatFile'] - agg['NestedFile']) / agg['FlatFile']) * 100

    # Sort logically so the X-axis flows smoothly from 10^2 up to 10^8
    agg = agg.sort_values(by=['total_elements', 'base_rows'])

    # Create a clean X-axis label combining Order of Magnitude and Shape
    agg['x_label'] = "10^" + agg['order_of_magnitude'].astype(str) + "\n(" + agg['base_rows'].astype(str) + "x" + agg['nested_rows'].astype(str) + ")"

    # 3. Generate a separate graph for each compression algorithm
    algorithms = agg['compression'].unique()
    
    for algo in algorithms:
        print(f"Generating graph for: {algo.upper()}...")
        subset = agg[agg['compression'] == algo]
        
        # Determine bar colors (Green for > 0, Red for < 0)
        colors = ['#2ca02c' if val >= 0 else '#d62728' for val in subset['percent_diff']]
        
        # Create wide figure to accommodate all configurations
        plt.figure(figsize=(18, 8))
        
        bars = plt.bar(subset['x_label'], subset['percent_diff'], color=colors, edgecolor='black', alpha=0.85)
        
        # Add a solid horizontal line at 0%
        plt.axhline(0, color='black', linewidth=1.5, linestyle='-')
        
        # Add a subtle grid
        plt.grid(axis='y', linestyle='--', alpha=0.6)
        
        # Labels and Title
        plt.title(f"Nested vs Flat Performance Difference — Algorithm: {algo.upper()}\n(Positive % = Nested is Faster | Negative % = Nested is Slower)", fontsize=16, fontweight='bold', pad=20)
        plt.xlabel("Order of Magnitude\n(Base Rows × Nested Rows)", fontsize=13, fontweight='bold', labelpad=15)
        plt.ylabel("Performance Difference (%)", fontsize=13, fontweight='bold')
        
        # Rotate X-axis labels to prevent overlapping
        plt.xticks(rotation=45, ha='right', fontsize=10)
        plt.yticks(fontsize=11)
        
        # Format Y-axis tick labels to show percentage sign
        current_yticks = plt.gca().get_yticks()
        plt.gca().set_yticklabels([f"{y:+.0f}%" for y in current_yticks])

        # Add data labels directly on the bars
        for bar in bars:
            height = bar.get_height()
            label_y = height + (1 if height >= 0 else -3) # Offset label slightly based on direction
            plt.text(bar.get_x() + bar.get_width()/2., label_y,
                     f'{height:+.1f}%', 
                     ha='center', va='bottom' if height >= 0 else 'top', 
                     fontsize=9, rotation=90)

        plt.tight_layout()
        
        # Save plot
        safe_algo_name = algo.replace(" ", "_")
        output_file = os.path.join(OUTPUT_DIR, f"{safe_algo_name}_performance.png")
        plt.savefig(output_file, dpi=200)
        plt.close()

    print(f"\n✅ All algorithm graphs saved to '{OUTPUT_DIR}/'")

if __name__ == "__main__":
    generate_algorithm_graphs()

Generating graph for: BROTLI...


/tmp/ipykernel_294879/2687322481.py:78: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  plt.gca().set_yticklabels([f"{y:+.0f}%" for y in current_yticks])


Generating graph for: GZIP...


/tmp/ipykernel_294879/2687322481.py:78: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  plt.gca().set_yticklabels([f"{y:+.0f}%" for y in current_yticks])


Generating graph for: LZ4...


...

Generating graph for: UNCOMPRESSED...


/tmp/ipykernel_294879/2687322481.py:78: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  plt.gca().set_yticklabels([f"{y:+.0f}%" for y in current_yticks])


Generating graph for: ZSTD...


/tmp/ipykernel_294879/2687322481.py:78: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  plt.gca().set_yticklabels([f"{y:+.0f}%" for y in current_yticks])



✅ All algorithm graphs saved to 'algorithm_percentage_graphs/'


In [9]:
import os
import gc
import time
import sqlite3
import subprocess
from datetime import datetime
import pyarrow.parquet as pq
from nested_pandas.datasets import generate_data

compressions = [None, "snappy", "gzip", "brotli", "lz4", "zstd"]


def GENERATE_DATA(b, n, file_path, algo, is_flat):
    """Generates synthetic nested or flat data and writes it to a Parquet file."""
    nf = generate_data(b, n, seed=1)[["nested"]]
    
    if is_flat:
        # .to_lists() already returns an object that can be saved directly to Parquet
        df = nf["nested"].to_lists()
        df.to_parquet(file_path, compression=algo)
    else:
        # Save as a native nested Parquet structure
        nf.to_parquet(file_path, compression=algo)


def SAVE_ALGO_TO_SQLITE3(sqlite3_path, file_obj, algo_key):
    """Saves individual benchmark runs for synthetic algorithms to the database."""
    conn = sqlite3.connect(sqlite3_path)
    cursor = conn.cursor()

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS benchmark_results (
            file_type TEXT,
            base_rows INTEGER,
            nested_rows INTEGER,
            compression TEXT,
            file_size_bytes INTEGER,
            run_number INTEGER,
            run_timestamp DATETIME,
            run_time_sec REAL
        )
    """)

    file_type = "FlatFile" if file_obj.is_flat else "NestedFile"
    file_size = file_obj.file_sizes.get(algo_key, 0)

    if algo_key in file_obj.benchmarking_times:
        for run_data in file_obj.benchmarking_times[algo_key]:
            run_number, timestamp, run_time = run_data

            cursor.execute(
                """
                INSERT INTO benchmark_results 
                (file_type, base_rows, nested_rows, compression, file_size_bytes, run_number, run_timestamp, run_time_sec)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?)
            """,
                (
                    file_type,
                    file_obj.b,
                    file_obj.n,
                    algo_key,
                    file_size,
                    run_number,
                    timestamp,
                    run_time,
                ),
            )

    conn.commit()
    conn.close()


def SAVE_CUSTOM_TO_SQLITE3(sqlite3_path, file_name, file_size, run_number, timestamp, run_time):
    """Saves benchmark runs for a specific custom file into a separate table."""
    conn = sqlite3.connect(sqlite3_path)
    cursor = conn.cursor()

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS custom_file_benchmarks (
            file_name TEXT,
            file_size_bytes INTEGER,
            run_number INTEGER,
            run_timestamp DATETIME,
            run_time_sec REAL
        )
    """)

    cursor.execute(
        """
        INSERT INTO custom_file_benchmarks 
        (file_name, file_size_bytes, run_number, run_timestamp, run_time_sec)
        VALUES (?, ?, ?, ?, ?)
    """,
        (file_name, file_size, run_number, timestamp, run_time)
    )

    conn.commit()
    conn.close()


class File:
    def __init__(self, b: int, n: int, path: str) -> None:
        self.b = b
        self.n = n
        self.order = b * n
        self.file_path = path
        self.file_exists = False

        self.file_sizes = {}
        self.benchmarking_times = {
            "None": [],
            "snappy": [],
            "gzip": [],
            "brotli": [],
            "lz4": [],
            "zstd": [],
        }

        self.current_algo = ""
        self.is_flat = False
        
        # Flag to control file deletion
        self.delete_after_benchmark = True 

    def generate_data(self, algo):
        GENERATE_DATA(self.b, self.n, self.file_path, algo, self.is_flat)
        self.file_exists = os.path.exists(self.file_path)

        algo_key = "None" if algo is None else algo
        self.file_sizes[algo_key] = (
            os.path.getsize(self.file_path) if self.file_exists else 0
        )

    def pre_benchmark(self):
        """Loads the file into the OS page cache using vmtouch."""
        if self.file_exists:
            subprocess.run(
                ["vmtouch", "-t", self.file_path], check=True, stdout=subprocess.DEVNULL
            )

    def run_benchmark_loop(self, db_path, num_runs=10):
        """Executes the complete granular lifecycle per algorithm sequentially."""
        for algo in compressions:
            self.current_algo = "None" if algo is None else algo
            print(f"      -> Processing Compression: {self.current_algo.upper()}")

            # 1 & 2. Choose algorithm & generate file
            self.generate_data(algo)

            # 3. Load file completely into OS page cache
            self.pre_benchmark()

            # 4. Benchmark the file for the specified number of runs
            for run_idx in range(1, num_runs + 1):
                run_timestamp = datetime.now().isoformat()

                start_time = time.perf_counter()
                table = pq.read_table(self.file_path, use_threads=True)
                run_time = time.perf_counter() - start_time

                del table
                gc.collect()

                self.benchmarking_times[self.current_algo].append(
                    (run_idx, run_timestamp, run_time)
                )

            # 5 & 6. Clear cache & delete file immediately after benchmarking
            self.post_benchmark()

            # 7. Save this specific algorithm's run data immediately to SQL
            SAVE_ALGO_TO_SQLITE3(db_path, self, self.current_algo)

    def post_benchmark(self):
        """Evicts the file from the OS page cache and optionally deletes it from disk."""
        if self.file_exists:
            subprocess.run(
                ["vmtouch", "-e", self.file_path], check=True, stdout=subprocess.DEVNULL
            )
            
            # Only delete if the flag is True
            if self.delete_after_benchmark:
                os.remove(self.file_path)
                self.file_exists = False


class FlatFile(File):
    def __init__(self, b: int, n: int, path: str) -> None:
        super().__init__(b, n, path)
        self.is_flat = True


class NestedFile(File):
    def __init__(self, b: int, n: int, path: str) -> None:
        super().__init__(b, n, path)
        self.is_flat = False


class CustomFileBenchmark:
    """A standalone class to benchmark ANY existing Parquet file without assumptions."""
    
    def __init__(self, path: str) -> None:
        self.file_path = path
        self.file_name = os.path.basename(path)
        self.benchmarking_times = []
        
        if not os.path.exists(self.file_path):
            raise FileNotFoundError(f"Custom file not found at: {self.file_path}")
            
        self.file_size = os.path.getsize(self.file_path)

    def pre_benchmark(self):
        """Loads the custom file into the OS page cache."""
        subprocess.run(
            ["vmtouch", "-t", self.file_path], check=True, stdout=subprocess.DEVNULL
        )

    def run_benchmark(self, db_path, num_runs=50):
        print(f"\n--- Benchmarking Custom File: {self.file_name} ---")
        
        # 1. Load into OS Cache
        self.pre_benchmark()

        # 2. Run read benchmarks
        for run_idx in range(1, num_runs + 1):
            run_timestamp = datetime.now().isoformat()
            
            start_time = time.perf_counter()
            table = pq.read_table(self.file_path, use_threads=True)
            run_time = time.perf_counter() - start_time
            
            del table
            gc.collect()

            # Save directly to the dedicated SQL table
            SAVE_CUSTOM_TO_SQLITE3(db_path, self.file_name, self.file_size, run_idx, run_timestamp, run_time)

        # 3. Evict from OS Cache (Does NOT delete file)
        self.post_benchmark()
        print(f" [✓] Completed {num_runs} runs on {self.file_name} and saved to 'custom_file_benchmarks' table.\n")

    def post_benchmark(self):
        """Evicts the custom file from the OS page cache."""
        subprocess.run(
            ["vmtouch", "-e", self.file_path], check=True, stdout=subprocess.DEVNULL
        )


# ==========================================
# Main Execution Pipeline
# ==========================================
if __name__ == "__main__":
    db_path = "benchmarks.db"

    # ==========================================
    # Example: Benchmarking a Custom File
    # ==========================================
    custom_file_path = "./part0.snappy.parquet"
    
    if os.path.exists(custom_file_path):
        custom_bench = CustomFileBenchmark(path=custom_file_path)
        custom_bench.run_benchmark(db_path, num_runs=10)


--- Benchmarking Custom File: part0.snappy.parquet ---
 [✓] Completed 10 runs on part0.snappy.parquet and saved to 'custom_file_benchmarks' table.



In [11]:
import os
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DB_PATH = "benchmarks.db"
OUTPUT_IMG = "custom_file_benchmark_results.png"

def plot_custom_benchmarks():
    if not os.path.exists(DB_PATH):
        print(f"Error: Database not found at {DB_PATH}.")
        return

    # 1. Load Data
    conn = sqlite3.connect(DB_PATH)
    try:
        df = pd.read_sql_query("SELECT * FROM custom_file_benchmarks", conn)
    except sqlite3.OperationalError:
        print("Error: Table 'custom_file_benchmarks' does not exist yet. Run the benchmark script first.")
        conn.close()
        return
    finally:
        conn.close()

    if df.empty:
        print("No custom benchmark data found in the database.")
        return

    print(f"Loaded {len(df)} custom file benchmark runs. Generating plots...")

    # 2. Setup the Plot Canvas (1 row, 2 columns)
    sns.set_theme(style="whitegrid")
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Calculate file sizes in MB for the labels
    file_sizes = df.groupby('file_name')['file_size_bytes'].first() / (1024 * 1024)
    df['file_label'] = df['file_name'].apply(lambda x: f"{x}\n({file_sizes[x]:.1f} MB)")

    # ---------------------------------------------------------
    # GRAPH 1: Line Plot (Sequential Run Times)
    # ---------------------------------------------------------
    sns.lineplot(
        data=df, 
        x='run_number', 
        y='run_time_sec', 
        hue='file_label', 
        marker='o', 
        linewidth=2,
        ax=axes[0]
    )
    
    axes[0].set_title("Sequential Read Speeds (Warmup & Variance)", fontsize=14, pad=15, fontweight='bold')
    axes[0].set_xlabel("Run Number", fontsize=12)
    axes[0].set_ylabel("Read Time (Seconds)", fontsize=12)
    
    # Force X-axis to show integers for run numbers
    axes[0].xaxis.get_major_locator().set_params(integer=True)

    # ---------------------------------------------------------
    # GRAPH 2: Box Plot (Statistical Distribution)
    # ---------------------------------------------------------
    sns.boxplot(
        data=df, 
        x='file_label', 
        y='run_time_sec', 
        hue='file_label', 
        showmeans=True,
        meanprops={"marker": "D", "markerfacecolor": "white", "markeredgecolor": "black"},
        ax=axes[1]
    )
    
    axes[1].set_title("Statistical Distribution of Read Times", fontsize=14, pad=15, fontweight='bold')
    axes[1].set_xlabel("Custom File Name", fontsize=12)
    axes[1].set_ylabel("Read Time (Seconds)", fontsize=12)
    axes[1].tick_params(axis='x', rotation=15)
    
    # Clean up legends
    axes[0].legend(title='File Name')
    if axes[1].get_legend():
        axes[1].get_legend().remove() # Remove redundant legend on the box plot

    # 3. Render and Save
    plt.tight_layout()
    plt.savefig(OUTPUT_IMG, dpi=300)
    plt.close()
    
    print(f"✅ Plot successfully saved as '{OUTPUT_IMG}'")

if __name__ == "__main__":
    plot_custom_benchmarks()

Loaded 10 custom file benchmark runs. Generating plots...
✅ Plot successfully saved as 'custom_file_benchmark_results.png'
